# 06 Model and Threshold Selection

Goal: keep the current best feature set (`Curated raw + engineered`) and compare several available scikit-learn models. Thresholds are selected on a validation split from the training data. The holdout test split is evaluated once at the end.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

import pandas as pd

from sklearn.ensemble import (
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

from src.features import BASIC_APPLICATION_FEATURES, create_features
from src.preprocessing import build_preprocessor
from src.thresholding import (
    evaluate_probabilities,
    find_best_threshold,
    get_positive_probabilities,
)


In [2]:
RANDOM_STATE = 42
TEST_SIZE = 0.2
VALIDATION_SIZE = 0.2
MIN_PRECISION_TARGET = 0.20

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

train_df = pd.read_csv(
    RAW_DATA_DIR / "application_train.csv"
)

y = train_df["TARGET"].copy()

X = train_df[BASIC_APPLICATION_FEATURES].copy()
X = create_features(X)

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full,
    y_train_full,
    test_size=VALIDATION_SIZE,
    stratify=y_train_full,
    random_state=RANDOM_STATE,
)

{
    "full_data_shape": X.shape,
    "train_shape": X_train.shape,
    "valid_shape": X_valid.shape,
    "test_shape": X_test.shape,
    "train_default_rate": y_train.mean(),
    "valid_default_rate": y_valid.mean(),
    "test_default_rate": y_test.mean(),
}


{'full_data_shape': (307511, 32),
 'train_shape': (196806, 32),
 'valid_shape': (49202, 32),
 'test_shape': (61503, 32),
 'train_default_rate': np.float64(0.08072924605957135),
 'valid_default_rate': np.float64(0.0807284256737531),
 'test_default_rate': np.float64(0.08072776937710356)}

In [3]:
def make_pipeline(model, scale_numeric=False, one_hot_sparse_output=True):
    numeric_columns = X_train.select_dtypes(include="number").columns.tolist()
    categorical_columns = X_train.select_dtypes(exclude="number").columns.tolist()

    preprocessor = build_preprocessor(
        numeric_columns=numeric_columns,
        categorical_columns=categorical_columns,
        scale_numeric=scale_numeric,
        one_hot_sparse_output=one_hot_sparse_output,
    )

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )


model_candidates = {
    "Logistic Regression": make_pipeline(
        LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        ),
        scale_numeric=True,
    ),
    "Random Forest current": make_pipeline(
        RandomForestClassifier(
            n_estimators=200,
            min_samples_split=2,
            min_samples_leaf=10,
            max_features="log2",
            max_depth=None,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    ),
    "Random Forest tuned shallow": make_pipeline(
        RandomForestClassifier(
            n_estimators=200,
            min_samples_split=10,
            min_samples_leaf=5,
            max_features="log2",
            max_depth=12,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    ),
    "Extra Trees": make_pipeline(
        ExtraTreesClassifier(
            n_estimators=200,
            min_samples_leaf=10,
            max_features="log2",
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    ),
    "Hist Gradient Boosting": make_pipeline(
        HistGradientBoostingClassifier(
            learning_rate=0.05,
            max_iter=200,
            max_leaf_nodes=31,
            l2_regularization=0.1,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        ),
        one_hot_sparse_output=False,
    ),
}

list(model_candidates)


['Logistic Regression',
 'Random Forest current',
 'Random Forest tuned shallow',
 'Extra Trees',
 'Hist Gradient Boosting']

In [4]:
selection_results = []
fitted_validation_models = {}

for model_name, pipeline in model_candidates.items():
    print(f"Fitting: {model_name}")

    pipeline.fit(X_train, y_train)
    valid_probabilities = get_positive_probabilities(pipeline, X_valid)

    default_metrics = evaluate_probabilities(
        y_true=y_valid,
        probabilities=valid_probabilities,
        threshold=0.5,
    )

    best_f1_threshold = find_best_threshold(
        y_true=y_valid,
        probabilities=valid_probabilities,
    )

    best_precision_threshold = find_best_threshold(
        y_true=y_valid,
        probabilities=valid_probabilities,
        min_precision=MIN_PRECISION_TARGET,
    )

    tuned_metrics = evaluate_probabilities(
        y_true=y_valid,
        probabilities=valid_probabilities,
        threshold=best_f1_threshold["threshold"],
    )

    selection_results.append({
        "model": model_name,
        "validation_roc_auc": tuned_metrics["roc_auc"],
        "validation_average_precision": tuned_metrics["average_precision"],
        "default_threshold_f1": default_metrics["f1_class_1"],
        "best_f1_threshold": best_f1_threshold["threshold"],
        "best_f1_precision": tuned_metrics["precision_class_1"],
        "best_f1_recall": tuned_metrics["recall_class_1"],
        "best_f1": tuned_metrics["f1_class_1"],
        "precision_target_threshold": best_precision_threshold["threshold"],
        "precision_target_validation_precision": best_precision_threshold[
            "validation_precision_class_1"
        ],
        "precision_target_validation_recall": best_precision_threshold[
            "validation_recall_class_1"
        ],
    })

    fitted_validation_models[model_name] = pipeline

selection_df = pd.DataFrame(selection_results).sort_values(
    ["best_f1", "validation_roc_auc"],
    ascending=False,
).reset_index(drop=True)

selection_df


Fitting: Logistic Regression


Fitting: Random Forest current


Fitting: Random Forest tuned shallow


Fitting: Extra Trees


Fitting: Hist Gradient Boosting


,model,validation_roc_auc,validation_average_precision,default_threshold_f1,best_f1_threshold,best_f1_precision,best_f1_recall,best_f1,precision_target_threshold,precision_target_validation_precision,precision_target_validation_recall
0,Hist Gradient Boosting,0.757085,0.237036,0.269402,0.658010,0.233276,0.443353,0.305703,0.591793,0.200019,0.542800
1,Random Forest current,0.743612,0.222539,0.289260,0.444572,0.217898,0.446878,0.292953,0.405177,0.200215,0.515609
2,Random Forest tuned shallow,0.742576,0.217561,0.276678,0.574532,0.216952,0.437563,0.290078,0.541954,0.200061,0.496727
3,Logistic Regression,0.741156,0.217104,0.254009,0.663440,0.223953,0.405337,0.288505,0.612845,0.200163,0.493958
4,Extra Trees,0.739272,0.215776,0.272604,0.566649,0.211746,0.439325,0.285761,0.547328,0.200805,0.477341


In [5]:
selected_model_name = selection_df.loc[0, "model"]
selected_threshold = selection_df.loc[0, "best_f1_threshold"]

selected_model_name, selected_threshold


('Hist Gradient Boosting', np.float64(0.6580095512213893))

In [6]:
def make_fresh_selected_pipeline(model_name):
    if model_name == "Logistic Regression":
        return make_pipeline(
            LogisticRegression(
                max_iter=3000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            ),
            scale_numeric=True,
        )

    if model_name == "Random Forest current":
        return make_pipeline(
            RandomForestClassifier(
                n_estimators=200,
                min_samples_split=2,
                min_samples_leaf=10,
                max_features="log2",
                max_depth=None,
                class_weight="balanced_subsample",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )
        )

    if model_name == "Random Forest tuned shallow":
        return make_pipeline(
            RandomForestClassifier(
                n_estimators=200,
                min_samples_split=10,
                min_samples_leaf=5,
                max_features="log2",
                max_depth=12,
                class_weight="balanced_subsample",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )
        )

    if model_name == "Extra Trees":
        return make_pipeline(
            ExtraTreesClassifier(
                n_estimators=200,
                min_samples_leaf=10,
                max_features="log2",
                class_weight="balanced",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )
        )

    if model_name == "Hist Gradient Boosting":
        return make_pipeline(
            HistGradientBoostingClassifier(
                learning_rate=0.05,
                max_iter=200,
                max_leaf_nodes=31,
                l2_regularization=0.1,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            ),
            one_hot_sparse_output=False,
        )

    raise ValueError(f"Unknown model: {model_name}")


final_model = make_fresh_selected_pipeline(selected_model_name)
final_model.fit(X_train_full, y_train_full)

test_probabilities = get_positive_probabilities(final_model, X_test)

test_default_threshold = evaluate_probabilities(
    y_true=y_test,
    probabilities=test_probabilities,
    threshold=0.5,
)

test_tuned_threshold = evaluate_probabilities(
    y_true=y_test,
    probabilities=test_probabilities,
    threshold=selected_threshold,
)

final_comparison = pd.DataFrame([
    {
        "model": selected_model_name,
        "threshold_strategy": "default_0.5",
        **test_default_threshold,
    },
    {
        "model": selected_model_name,
        "threshold_strategy": "validation_best_f1",
        **test_tuned_threshold,
    },
])

final_comparison


,model,threshold_strategy,threshold,roc_auc,average_precision,accuracy,precision_class_1,recall_class_1,f1_class_1
0,Hist Gradient Boosting,default_0.5,0.50000,0.762573,0.252267,0.706990,0.16987,0.676536,0.271555
1,Hist Gradient Boosting,validation_best_f1,0.65801,0.762573,0.252267,0.842138,0.24329,0.452769,0.316508


In [7]:
from sklearn.metrics import confusion_matrix, classification_report

for threshold_strategy, threshold in [
    ("default_0.5", 0.5),
    ("validation_best_f1", selected_threshold),
]:
    predictions = (test_probabilities >= threshold).astype(int)

    print(threshold_strategy)
    print("threshold:", threshold)
    print(confusion_matrix(y_test, predictions))
    print(classification_report(y_test, predictions, zero_division=0))


default_0.5
threshold: 0.5
[[40123 16415]
 [ 1606  3359]]
              precision    recall  f1-score   support

           0       0.96      0.71      0.82     56538
           1       0.17      0.68      0.27      4965

    accuracy                           0.71     61503
   macro avg       0.57      0.69      0.54     61503
weighted avg       0.90      0.71      0.77     61503

validation_best_f1
threshold: 0.6580095512213893
[[49546  6992]
 [ 2717  2248]]
              precision    recall  f1-score   support

           0       0.95      0.88      0.91     56538
           1       0.24      0.45      0.32      4965

    accuracy                           0.84     61503
   macro avg       0.60      0.66      0.61     61503
weighted avg       0.89      0.84      0.86     61503



## Interpretation Template

Use `selection_df` to choose the best model and threshold using validation data. Use `final_comparison` to report the final holdout performance. If the tuned threshold improves class-1 F1, use it for classification decisions; keep ROC-AUC and average precision for ranking quality.
